In [26]:
import sys
# Add the directory containing your file to the system path
sys.path.insert(1, '/kaggle/input/datasets/naynoona/helpers')

In [27]:
import helpers
import numpy as np
import os
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
import torch
from sklearn.metrics import classification_report
import json

In [28]:
# I will simply load the entire dataset into a data loader (for batch processing)
train_dataset_path = r"/kaggle/input/datasets/naynoona/cmr-multi/CMR-MULTI/CINE_MULTI/4CH_TR/image"
mask_dataset_path = r"/kaggle/input/datasets/naynoona/cmr-multi/CMR-MULTI/CINE_MULTI/4CH_TR/anno"


all_images = helpers.get_all_np_images(train_dataset_path)
zscaler, clipped_images = helpers.train_scaler(all_images)

test_data_percentage, val_data_percentage = 0.3, 0.15
mri_dataset, test_mri_dataset, val_mri_dataset = helpers.create_2d_dataset(train_dataset_path, mask_dataset_path, scaler=zscaler, test_data_percentage=test_data_percentage, val_data_percentage=val_data_percentage)


batch_size = 20
mri_dataloader = DataLoader(mri_dataset, batch_size=batch_size)
test_mri_dataloader = DataLoader(test_mri_dataset, batch_size=batch_size)
val_mri_dataloader  = DataLoader(val_mri_dataset, batch_size=batch_size)

classes = list(helpers.load_labels(Problem_Type="CINE", Image_Type="4CH").keys())
print(len(classes))
print(classes)

100%|██████████| 105/105 [00:33<00:00,  3.16it/s]


finished clipping 105 images
finished flattening 105 images to 217025575 pixels
finished training the scaler
All dataset is of length torch.Size([8285, 1, 169, 155]) 
With Masks of torch.Size([8285, 1, 169, 155])
Splitting — train: 4558  val: 1242  test: 2485
6
[0, 1, 2, 3, 4, 5]


In [29]:
class FCN(nn.Module):
    def __init__(self, in_channels, num_classes):
        """
        FCN with learned transposed convolution upsampling.
        Each decoder stage: ConvTranspose2d (upsample ×2) → Conv2d (refine)
        """
        super(FCN, self).__init__()

        # ── Encoder ──────────────────────────────────────────────────────────
        self.conv1_1 = nn.Conv2d(in_channels, 32,  kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(32,          32,  kernel_size=3, padding=1)
        self.pool1   = nn.MaxPool2d(2, 2)                               # H/2

        self.conv2_1 = nn.Conv2d(32,  64, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(64,  64, kernel_size=3, padding=1)
        self.pool2   = nn.MaxPool2d(2, 2)                               # H/4

        self.conv3_1 = nn.Conv2d(64,  128, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool3   = nn.MaxPool2d(2, 2)                               # H/8

        # ── Bottleneck ───────────────────────────────────────────────────────
        self.conv4_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

        # ── Decoder stage 1: H/8 → H/4 ──────────────────────────────────────
        # ConvTranspose doubles spatial size; +128 because we concat skip_8x
        self.up1     = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1_1  = nn.Conv2d(128 + 128, 128, kernel_size=3, padding=1)
        self.dec1_2  = nn.Conv2d(128,       128, kernel_size=3, padding=1)

        # ── Decoder stage 2: H/4 → H/2 ──────────────────────────────────────
        # +64 because we concat skip_4x
        self.up2     = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2_1  = nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1)
        self.dec2_2  = nn.Conv2d(64,      64, kernel_size=3, padding=1)

        # ── Decoder stage 3: H/2 → H ─────────────────────────────────────────
        # No skip here (pool1 features are very shallow, marginal benefit)
        self.up3     = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec3_1  = nn.Conv2d(32, 32, kernel_size=3, padding=1)

        # ── Final classifier head ─────────────────────────────────────────────
        self.classifier = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        orig_size = x.shape[2:]

        # ── Encoder ──────────────────────────────────────────────────────────
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.pool1(x)                   # H/2

        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        skip_4x = x                         # save /4 features for skip
        x = self.pool2(x)                   # H/4

        x = F.relu(self.conv3_1(x))
        x = F.relu(self.conv3_2(x))
        skip_8x = x                         # save /8 features for skip
        x = self.pool3(x)                   # H/8

        # ── Bottleneck ───────────────────────────────────────────────────────
        x = F.relu(self.conv4_1(x))
        x = F.relu(self.conv4_2(x))

        # ── Decoder ──────────────────────────────────────────────────────────
        # Stage 1: learned upsample + concat skip_8x + refine
        x = self.up1(x)                                        # H/8 → H/4
        x = self._match_and_cat(x, skip_8x)                   # handle odd sizes
        x = F.relu(self.dec1_1(x))
        x = F.relu(self.dec1_2(x))

        # Stage 2: learned upsample + concat skip_4x + refine
        x = self.up2(x)                                        # H/4 → H/2
        x = self._match_and_cat(x, skip_4x)
        x = F.relu(self.dec2_1(x))
        x = F.relu(self.dec2_2(x))

        # Stage 3: learned upsample + refine
        x = self.up3(x)                                        # H/2 → H
        x = F.relu(self.dec3_1(x))

        # Final resize handles any residual off-by-one from odd input dims
        x = F.interpolate(x, size=orig_size, mode='bilinear', align_corners=False)

        return self.classifier(x)           # (N, num_classes, H, W)

    @staticmethod
    def _match_and_cat(upsampled, skip):
        """
        Crop or pad upsampled to match skip's spatial size before concat.
        Needed because ConvTranspose2d on odd dimensions can be off by 1px.
        """
        if upsampled.shape[2:] != skip.shape[2:]:
            upsampled = F.interpolate(upsampled, size=skip.shape[2:],
                                      mode='bilinear', align_corners=False)
        return torch.cat([upsampled, skip], dim=1)
    

hist_fcn = {'train_acc': [], 'train_dice': [], 'train_f1': [], 'val_acc': [], 'val_dice': [], 'val_f1': []}

In [30]:
# ── Helper: run one full pass over a dataloader and return metrics ────────────
 
def run_eval(model, dataloader, device):
    """
    Runs model in eval mode over *all* batches in `dataloader`.
    Returns (dice, accuracy, f1) computed on the full set — not per-batch averages.
    """
    model.eval()
    all_preds   = []
    all_targets = []
 
    with torch.no_grad():
        for data, targets in dataloader:
            data    = data.to(device)
            targets = targets.squeeze(1).long().to(device)
            scores  = model(data)
            preds   = torch.argmax(scores, dim=1)   # (N, H, W)
 
            all_preds.append(preds.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
 
    all_preds   = np.concatenate(all_preds,   axis=0)  # (total_N, H, W)
    all_targets = np.concatenate(all_targets, axis=0)
 
    return helpers.validate(all_targets, all_preds, num_classes=len(classes))
 

In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using {device}")

model = FCN(in_channels=1, num_classes=len(classes)).to(device)
print(model)


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        num_classes = logits.shape[1]

        # Convert logits to probabilities over class dimension
        probs = torch.softmax(logits, dim=1)  # (N, C, H, W)

        # One-hot encode targets: (N, H, W) -> (N, C, H, W)
        targets_one_hot = torch.zeros_like(probs)  # (N, C, H, W)
        targets_one_hot.scatter_(1, targets.unsqueeze(1), 1)  # fill class channels

        # Flatten spatial dims, keep class dim: (N, C, H*W)
        probs = probs.view(probs.shape[0], num_classes, -1)
        targets_one_hot = targets_one_hot.view(targets_one_hot.shape[0], num_classes, -1)

        # Compute Dice per class, then average
        intersection = (probs * targets_one_hot).sum(dim=2)  # (N, C)
        dice = (2. * intersection + self.smooth) / (
            probs.sum(dim=2) + targets_one_hot.sum(dim=2) + self.smooth
        )  # (N, C)

        dice = dice[:, 1:]   # ignore class 0
        return 1 - dice.mean()

# Define the loss function
class_weights = torch.tensor([0.1, 1.0, 1.0, 1.0, 1.0, 1.0], dtype=torch.float).to(device)
ce_loss = nn.CrossEntropyLoss(weight=class_weights)


def combined_loss(logits, targets):
    return ce_loss(logits, targets) + DiceLoss()(logits, targets)

criterion = combined_loss


# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.00005)


num_epochs = 75
for epoch in range(num_epochs):

    # Choose some random batches for validation
    #n_samples_val = int(val_data_percentage * len(mri_dataloader))
    #random_batch_indices = np.random.randint(0, len(mri_dataloader), n_samples_val // batch_size)
    #print(random_batch_indices)

    model.train()
    print(f"Epoch [{epoch + 1}/{num_epochs}]")

    running_loss = 0.0
    total_samples = 0
    train_preds_all = torch.tensor([]).to(device)
    train_targets_all = torch.tensor([]).to(device)

    for batch_index, (data, targets) in enumerate(tqdm(mri_dataloader)):
        data = data.to(device)
        targets = targets.squeeze(1).long().to(device)
        scores = model(data)
        loss = criterion(scores, targets)

        preds = torch.argmax(scores, dim=1)  # (N, H, W)

        # ── Training batch ───────────────────────────────────────────────────
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size     = data.size(0)
        running_loss  += loss.item() * batch_size
        total_samples += batch_size

        # Accumulate predictions for epoch-level training metrics
        train_preds_all = torch.cat((train_preds_all, preds), dim=0).to(device)
        train_targets_all = torch.cat((train_targets_all, targets), dim=0).to(device)

    # ── End-of-epoch training metrics ────────────────────────────────────────
    epoch_loss = running_loss / total_samples
    #print(f"Epoch [{epoch + 1}] Loss: {epoch_loss:.4f}")

    val_dice, val_acc, val_f1 = run_eval(model, val_mri_dataloader, device)
    hist_fcn['val_acc'].append((epoch + 1, val_acc))
    hist_fcn['val_dice'].append((epoch + 1, val_dice))
    hist_fcn['val_f1'].append((epoch + 1, val_f1))

    # Concatenate all training batches: (total_N, H, W)
    train_preds_np   = train_preds_all.to('cpu').numpy()
    train_targets_np = train_targets_all.to('cpu').numpy()

    if(epoch % 10 == 0): #after each 10 epochs, write results so as not to lose them mid-execution
      with open("hist_fcn.json", "w") as f:
        json.dump(hist_fcn, f, indent=4) 

    train_dice, train_acc, train_f1 = helpers.validate(
        train_targets_np,
        train_preds_np,
        num_classes=len(classes)
    )

    hist_fcn['train_acc'].append((epoch + 1, train_acc))
    hist_fcn['train_f1'].append((epoch + 1, train_f1))
    hist_fcn['train_dice'].append((epoch + 1, train_dice))

    print(f"Epoch [{epoch + 1}] Train - dice: {train_dice:.4f}  acc: {train_acc:.4f}  f1: {train_f1:.4f}")




def generate_classification_report(model, dataloader_obj):
    model.eval()
    all_preds   = []
    all_targets = []

    with torch.no_grad():
        for images, labels in dataloader_obj:
            images = images.to(device)
            labels = labels.squeeze(1).long().to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.append(preds.cpu())
            all_targets.append(labels.cpu())
            #print(torch.unique(preds))

    all_preds   = torch.cat(all_preds).view(-1)
    all_targets = torch.cat(all_targets).view(-1)

    return classification_report(all_targets.numpy(), all_preds.numpy())


using cuda
FCN(
  (conv1_1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1_2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2_1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2_2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3_1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3_2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv4_1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4_2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (up1): ConvTranspose2d(256, 128, kernel_size=(2, 2), stride=(2, 2))
  (dec1_1): Conv2d

100%|██████████| 228/228 [00:33<00:00,  6.89it/s]


Epoch [1] Train - dice: 0.0444  acc: 0.5561  f1: 0.1628
Epoch [2/75]


100%|██████████| 228/228 [00:33<00:00,  6.83it/s]


Epoch [2] Train - dice: 0.2263  acc: 0.6997  f1: 0.3310
Epoch [3/75]


...

Epoch [73] Train - dice: 0.9075  acc: 0.9691  f1: 0.9206
Epoch [74/75]


100%|██████████| 228/228 [00:35<00:00,  6.41it/s]


Epoch [74] Train - dice: 0.9016  acc: 0.9674  f1: 0.9156
Epoch [75/75]


100%|██████████| 228/228 [00:35<00:00,  6.41it/s]


Epoch [75] Train - dice: 0.9016  acc: 0.9672  f1: 0.9155


In [32]:
with open("hist_fcn.json", "w") as f:
    json.dump(hist_fcn, f, indent=4)

In [33]:
test_dice, test_acc, test_f1 = run_eval(model, test_mri_dataloader, device)
print(f"\nTest — dice: {test_dice:.4f}  acc: {test_acc:.4f}  f1: {test_f1:.4f}")

print(generate_classification_report(model, test_mri_dataloader))


Test — dice: 0.6846  acc: 0.8742  f1: 0.7266
              precision    recall  f1-score   support

           0       0.95      0.92      0.94  24793216
           1       0.77      0.79      0.78   2294082
           2       0.48      0.55      0.51   1420128
           3       0.50      0.80      0.62   1403608
           4       0.70      0.68      0.69   1118066
           5       0.88      0.78      0.83   1505090

    accuracy                           0.87  32534190
   macro avg       0.71      0.75      0.73  32534190
weighted avg       0.89      0.87      0.88  32534190



In [ ]:
images, masks = next(iter(test_mri_dataloader))

images = images.to(device)
masks = masks.to(device)

masks = masks.squeeze(1).long()

model.eval()
with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)    

images = images.cpu()
masks = masks.cpu()
preds = preds.cpu()


import matplotlib.pyplot as plt

def visualize_sample(img, mask, pred, labels_map):
    figure,axis = plt.subplots(1,3,figsize=(15,10))

    axis[0].imshow(img.squeeze(), cmap='grey')
    axis[0].set_title("Input Image")

    Plot_Color_Op = axis[1].imshow(mask, cmap='hot')
    axis[1].set_title("Label Mask")

    axis[2].imshow(pred, cmap='hot')
    axis[2].set_title("Predicted Mask")

    cbar = figure.colorbar(Plot_Color_Op, ax=axis.ravel().tolist(), shrink=0.3, ticks=list(labels_map.keys()), label='Label Legend')
    cbar.ax.set_yticklabels(list(labels_map.values()))

labelled_classes = helpers.load_labels(Problem_Type="CINE", Image_Type="4CH")
for i in range(5):  # change number as you like
    visualize_sample(images[i], masks[i], preds[i], labelled_classes)



In [35]:
def average_by_epoch(metrics_dict):
    result = {}

    for metric, values in metrics_dict.items():
        epoch_sums = {}
        epoch_counts = {}

        # accumulate sums and counts
        for epoch, score in values:
            if(epoch in epoch_sums.keys()):
                epoch_sums[epoch] += score
                epoch_counts[epoch] += 1
            else:
                epoch_sums[epoch] = score
                epoch_counts[epoch] = 1

        # compute averages
        averaged = []
        for epoch in sorted(epoch_sums.keys()):
            avg = epoch_sums[epoch] / epoch_counts[epoch]
            averaged.append((epoch, avg))

        result[metric] = averaged

    return result


hist_fcn = average_by_epoch(hist_fcn)
with open("hist_fcn.json", "w") as f:
    json.dump(hist_fcn, f, indent=4)




In [ ]:
print(hist_fcn)
fig, ax = plt.subplots(3, 1, figsize=(9, 20))

train_epochs = [e for e, _ in hist_fcn['train_acc']]
train_acc = [v for _, v in hist_fcn['train_acc']]

val_epochs = [e for e, _ in hist_fcn['val_acc']]
val_acc = [v for _, v in hist_fcn['val_acc']]

ax[0].plot(train_epochs, train_acc, color='blue', label="Train Accuracy")
ax[0].plot(val_epochs, val_acc, color='red', label="Validation Accuracy")
ax[0].set_title("Accuracy Curve")
ax[0].legend()
#ax[0].show()



#fig, ax = plt.subplots(figsize=(9, 4.5))

train_epochs = [e for e, _ in hist_fcn['train_f1']]
train_f1 = [v for _, v in hist_fcn['train_f1']]

val_epochs = [e for e, _ in hist_fcn['val_f1']]
val_f1 = [v for _, v in hist_fcn['val_f1']]

ax[1].plot(train_epochs, train_f1, color='blue', label="Train F1")
ax[1].plot(val_epochs, val_f1, color='red', label="Validation F1")
ax[1].set_title("F1 Curve")
ax[1].legend()
#ax[1].show()


#fig, ax = plt.subplots(figsize=(9, 4.5))

train_epochs = [e for e, _ in hist_fcn['train_dice']]
train_dice = [v for _, v in hist_fcn['train_dice']]

val_epochs = [e for e, _ in hist_fcn['val_dice']]
val_dice = [v for _, v in hist_fcn['val_dice']]

ax[2].plot(train_epochs, train_dice, color='blue', label="Train dice")
ax[2].plot(val_epochs, val_dice, color='red', label="Validation dice")
ax[2].set_title("Dice Curve")
ax[2].legend()
#ax[2].show()